# Targeted Fixing Tool (Gold Labeling)

Use this notebook to manually fix labels in two stages:
1. **Critical Fixes:** 'Unknown' labels where the model failed completely.
2. **Ambiguous Refinement:** 'Error' labels or regex fallbacks that need human judgment.

In [6]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# --- Configuration ---
LABELS_PATH = '../data/labels/action_labels_llm_clean_refined.csv'

# Action Taxonomy
TAXONOMY = [
    "Locomotion",
    "Essential Operation",
    "Object Transfer",
    "Search",
    "Error / Correction",
    "Stationary",
    "Unknown" 
]

In [7]:
# --- Helper Functions & Class ---

def load_data():
    if not os.path.exists(LABELS_PATH):
        print("File not found!")
        return pd.DataFrame()
    return pd.read_csv(LABELS_PATH)

def save_row(df, index, new_action, status='gold'):
    df.at[index, 'action'] = new_action
    df.at[index, 'status'] = status
    df.to_csv(LABELS_PATH, index=False)
    # print(f"Saved row {index} as {new_action} ({status})")

def delete_row(df, index):
    df.at[index, 'status'] = 'deleted'
    df.to_csv(LABELS_PATH, index=False)
    # print(f"Marked row {index} as DELETED")

class LabelingTool:
    def __init__(self, df, mask, title="Labeling Tool"):
        self.df = df
        self.mask = mask
        self.fix_indices = df[mask].index.tolist()
        self.indices_to_process = []
        self.current_ptr = 0
        self.title = title
        
        # Components
        self.info_box = widgets.HTML()
        self.dropdown = widgets.Dropdown(description='Label:', options=TAXONOMY)
        self.save_btn = widgets.Button(description="SAVE (Gold)", button_style='success')
        self.delete_btn = widgets.Button(description="DELETE", button_style='danger')
        self.skip_btn = widgets.Button(description="Skip", button_style='')
        self.batch_input = widgets.IntText(value=50, description='Batch Size:')
        self.start_btn = widgets.Button(description=f"Start {title}")
        
        # Layouts
        self.controls = widgets.HBox([self.dropdown, self.save_btn, self.delete_btn, self.skip_btn])
        self.main_layout = widgets.VBox([self.info_box, self.controls])
        self.main_layout.layout.display = 'none'
        
        # Events
        self.start_btn.on_click(self.start_batch)
        self.save_btn.on_click(self.on_save)
        self.delete_btn.on_click(self.on_delete)
        self.skip_btn.on_click(self.on_skip)
        
    def display(self):
        print(f"--- {self.title} ---")
        print(f"Pool Size: {len(self.fix_indices)} rows")
        display(self.batch_input)
        display(self.start_btn)
        display(self.main_layout)
        
    def start_batch(self, b):
        limit = self.batch_input.value
        # Reload indices in case they changed (optional, but good practice)
        self.fix_indices = self.df[self.mask].index.tolist()
        self.indices_to_process = self.fix_indices[:limit]
        self.current_ptr = 0
        self.main_layout.layout.display = 'flex'
        self.show_sample()
        
    def show_sample(self):
        if self.current_ptr >= len(self.indices_to_process):
            self.info_box.value = "<h3>Batch completed!</h3>"
            self.controls.layout.display = 'none'
            return
        
        self.controls.layout.display = 'flex'
        row_idx = self.indices_to_process[self.current_ptr]
        row = self.df.loc[row_idx]
        
        self.info_box.value = f"""
        <div style="border:1px solid #ddd; padding:10px; margin-bottom:10px; background-color:#f9f9f9;">
            <div style="margin-bottom:10px; border-bottom:1px solid #ccc; padding-bottom:5px;">
                <strong>Progress:</strong> {self.current_ptr+1}/{len(self.indices_to_process)} 
                <span style="float:right; color:#666;">Total Pool: {len(self.fix_indices)}</span>
            </div>
            <p><strong>Row ID:</strong> {row_idx} | <strong>Time:</strong> {row['timestamp_sec']}s</p>
            <p><strong>Narration:</strong> <span style="color:blue; font-size:1.1em;">{row['narration_text']}</span></p>
            <p><strong>Current:</strong> {row['action']} <em>({row['status']})</em></p>
            <p><strong>Reasoning:</strong> {row['reasoning']}</p>
        </div>
        """
        self.dropdown.value = row['action'] if row['action'] in TAXONOMY else 'Unknown'
        
    def on_save(self, b):
        if self.current_ptr < len(self.indices_to_process):
            row_idx = self.indices_to_process[self.current_ptr]
            save_row(self.df, row_idx, self.dropdown.value, status='gold')
            self.current_ptr += 1
            self.show_sample()
            
    def on_delete(self, b):
        if self.current_ptr < len(self.indices_to_process):
            row_idx = self.indices_to_process[self.current_ptr]
            delete_row(self.df, row_idx)
            self.current_ptr += 1
            self.show_sample()
            
    def on_skip(self, b):
        if self.current_ptr < len(self.indices_to_process):
            self.current_ptr += 1
            self.show_sample()  

In [ ]:
# --- Load Data & Define Masks ---
df = load_data()

# 1. Critical Mask (Unknowns)
mask_critical = (
    (df['action'] == 'Unknown')
) & (df['status'] != 'gold') & (df['status'] != 'deleted')

# 2. Ambiguous Mask (Errors/Fallbacks)
mask_ambiguous = (
    (df['action'].astype(str).str.contains('Error', case=False, na=False)) | 
    (df['reasoning'].astype(str).str.contains('fallback', case=False, na=False))
) & (df['status'] != 'gold') & (df['status'] != 'deleted')


### Step 1: Critical Fixes
Fix rows where the model failed completely (Unknowns).

In [ ]:
tool_critical = LabelingTool(df, mask_critical, "Critical Fixes")
tool_critical.display()

### Step 2: Ambiguous Refinement
Review rows labeled as 'Error' or 'Regex Fallback' to decide between Object Transfer vs. Error.

In [ ]:
tool_ambiguous = LabelingTool(df, mask_ambiguous, "Ambiguous Refinement")
tool_ambiguous.display()

## Stage 2.5: LLM Strict Error/Correction mapping
Use Qwen2.5 (Text Only) with context to fix remaining errors. 
**Strictly forbids 'Error / Correction' in the output.**

In [4]:
import pandas as pd
from vllm import LLM, SamplingParams
import torch
import json
import re

# --- Config ---
INPUT_CSV = '../data/labels/action_labels_llm_clean_refined.csv'
OUTPUT_CSV = INPUT_CSV
MODEL_PATH = "Qwen/Qwen2.5-14B-Instruct-AWQ"

# Forced Taxonomy (NO ERROR/CORRECTION)
FORCED_TAXONOMY = [
    "Locomotion",
    "Essential Operation",
    "Object Transfer",
    "Search",
    "Stationary",
]

# STRICT PROMPT
SYSTEM_PROMPT_CONTEXT = f"""You are an expert at analyzing human behavior logs.
You will see a sequence of 4 actions (t-3, t-2, t-1, Current).
The Current Action is labeled as 'Error / Correction', which is INVALID for this task.
You MUST re-classify the action into one of the valid categories below.

Valid Taxonomy: {json.dumps(FORCED_TAXONOMY)}

Instructions:
- If it looks like a deliberate placement (e.g., dropping ONTO surface, putting down, repeating a movement for a specific purpose), label as 'Object Transfer'.
- If you think it could be something else, choose a label from the taxonomy.
- Do NOT output 'Error / Correction'.

Output JSON:
{{
  "new_label": "Category",
  "reasoning": "Why context matters..."
}}
"""

In [2]:
# --- Load Data ---
df_test = pd.read_csv(INPUT_CSV)
# Filter for Error/Correction ONLY
errors_to_check = df_test[
    (df_test['action'] == 'Error / Correction')
].index.tolist()

print(f"Found {len(errors_to_check)} rows to force-fix.")

# --- Initialize Model ---
# Only init if not already exists
try:
    llm
except NameError:
    llm = LLM(
        model=MODEL_PATH,
        gpu_memory_utilization=0.5, 
        enforce_eager=True,
        tensor_parallel_size=2
    )
    sampling_params = SamplingParams(temperature=0.1, max_tokens=512)

Found 468 rows to force-fix.
INFO 02-05 17:19:16 [utils.py:261] non-default args: {'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ'}
INFO 02-05 17:19:16 [model.py:541] Resolved architecture: Qwen2ForCausalLM
INFO 02-05 17:19:16 [model.py:1561] Using max model len 32768
INFO 02-05 17:19:18 [awq_marlin.py:162] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 02-05 17:19:18 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

INFO 02-05 17:19:18 [vllm.py:624] Asynchronous scheduling is enabled.
WARNING 02-05 17:19:18 [vllm.py:662] Enforce eager set, overriding optimization level to -O0
INFO 02-05 17:19:18 [vllm.py:762] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=3260309) INFO 02-05 17:19:19 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='Qwen/Qwen2.5-14B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-14B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disabl

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3260309) (Worker_TP0 pid=3260323) INFO 02-05 17:19:25 [default_loader.py:291] Loading weights took 2.52 seconds
(EngineCore_DP0 pid=3260309) (Worker_TP0 pid=3260323) INFO 02-05 17:19:27 [gpu_model_runner.py:4118] Model loading took 4.69 GiB memory and 5.205756 seconds
(EngineCore_DP0 pid=3260309) (Worker_TP0 pid=3260323) INFO 02-05 17:19:31 [gpu_worker.py:356] Available KV cache memory: 17.43 GiB
(EngineCore_DP0 pid=3260309) INFO 02-05 17:19:31 [kv_cache_utils.py:1307] GPU KV cache size: 190,400 tokens
(EngineCore_DP0 pid=3260309) INFO 02-05 17:19:31 [kv_cache_utils.py:1312] Maximum concurrency for 32,768 tokens per request: 5.81x
(EngineCore_DP0 pid=3260309) INFO 02-05 17:19:32 [core.py:272] init engine (profile, create kv cache, warmup model) took 4.06 seconds
(EngineCore_DP0 pid=3260309) INFO 02-05 17:19:32 [vllm.py:762] Cudagraph is disabled under eager mode
WARNING 02-05 17:19:32 [vllm.py:669] Inductor compilation was disabled by user settings, optimizations se

In [3]:
def get_context_prompt(df, idx, window=3):
    current_row = df.loc[idx]
    uid = current_row['video_uid']
    video_df = df[df['video_uid'] == uid].sort_values('timestamp_sec')
    video_df_indices = video_df.index.tolist()
    
    try:
        pos = video_df_indices.index(idx)
    except ValueError:
        return ""
    
    start_pos = max(0, pos - window)
    context_indices = video_df_indices[start_pos:pos]
    
    context_str = ""
    for i, ctx_idx in enumerate(context_indices):
        row = df.loc[ctx_idx]
        offset = len(context_indices) - i
        context_str += f"t-{offset}: {row['narration_text']} (Label: {row['action']})\n"
        
    return context_str

# --- Fast Batching Logic ---
print("Starting Fast Batch Correction...")

prompts = []
indices = []

# Prepare ALL prompts upfront
for idx in errors_to_check:
    row = df_test.loc[idx]
    context_str = get_context_prompt(df_test, idx)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_CONTEXT},
        {"role": "user", "content": f"Sequence:\n{context_str}\n\nCurrent Action: {row['narration_text']}\n\nOutput JSON:"}
    ]
    text_prompt = llm.get_tokenizer().apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text_prompt)
    indices.append(idx)

# Run Inference (vLLM handles batching)
print(f"Generating {len(prompts)} responses...")
outputs = llm.generate(prompts, sampling_params, use_tqdm=True)

# Process Results
changes_count = 0
for i, output in enumerate(outputs):
    idx = indices[i]
    text = output.outputs[0].text.strip()
    
    # Parse
    new_label = "Unknown"
    reasoning = ""
    try:
        code_block = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
        if code_block:
            data = json.loads(code_block.group(1))
        else:
            start = text.find('{')
            end = text.rfind('}') + 1
            data = json.loads(text[start:end])
        new_label = data.get('new_label', 'Unknown')
        reasoning = data.get('reasoning', '')
    except:
        pass
        
    # Validation: Ensure it's not Error/Correction
    if new_label == 'Error / Correction':
        new_label = 'Unknown' # Hard fallback

    if new_label in FORCED_TAXONOMY and new_label != df_test.loc[idx, 'action']:
        print(f"[Fix {idx}] {df_test.loc[idx, 'action']} -> {new_label}")
        df_test.at[idx, 'action'] = new_label
        df_test.at[idx, 'reasoning'] = f"[CtxFixed] {reasoning}"
        changes_count += 1

print(f"\nDone! Fixed {changes_count} rows.")
df_test.to_csv(OUTPUT_CSV, index=False)
print(f"Saved to {OUTPUT_CSV}")

Starting Fast Batch Correction...


Generating 468 responses...


Adding requests:   0%|          | 0/468 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/468 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[Fix 52] Error / Correction -> Object Transfer
[Fix 131] Error / Correction -> Object Transfer
[Fix 268] Error / Correction -> Object Transfer
[Fix 270] Error / Correction -> Object Transfer
[Fix 272] Error / Correction -> Object Transfer
[Fix 274] Error / Correction -> Object Transfer
[Fix 282] Error / Correction -> Object Transfer
[Fix 284] Error / Correction -> Object Transfer
[Fix 286] Error / Correction -> Object Transfer
[Fix 288] Error / Correction -> Object Transfer
[Fix 302] Error / Correction -> Object Transfer
[Fix 304] Error / Correction -> Object Transfer
[Fix 306] Error / Correction -> Object Transfer
[Fix 309] Error / Correction -> Object Transfer
[Fix 311] Error / Correction -> Object Transfer
[Fix 314] Error / Correction -> Object Transfer
[Fix 317] Error / Correction -> Object Transfer
[Fix 320] Error / Correction -> Object Transfer
[Fix 322] Error / Correction -> Object Transfer
[Fix 327] Error / Correction -> Object Transfer
[Fix 330] Error / Correction -> Object Tr

In [3]:
# --- Cleanup to free GPU ---
import gc
import torch

try:
    del llm
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("GPU memory released.")

GPU memory released.


In [2]:
# --- Final Verification ---
df_check = pd.read_csv(INPUT_CSV)
remaining = len(df_check[df_check['action'] == 'Error / Correction'])
print(f"Remaining Error / Correction labels: {remaining}")

KeyError: 'action'